In [8]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
import gensim.downloader as api
from datasets import load_dataset
from sklearn.metrics import classification_report
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

from task6.torch_utils.model_callbacks import EarlyStopping, ModelCheckpoint
from task6.utils.balance_dataset import augment_minority_classes
from task6.utils.prepare_data import prepare_data
from task6.utils.prepare_rnn_data import prepare_rnn_data

In [9]:
import torch
import torchvision

print("Torch version:", torch.__version__)
print("Torchvision version:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")


Torch version: 2.8.0+cu129
Torchvision version: 0.23.0+cu129
CUDA available: True
CUDA device: NVIDIA GeForce RTX 5080


In [10]:
dataset = load_dataset("google-research-datasets/go_emotions")
df = pd.DataFrame(dataset["train"])

In [11]:
df_test = pd.read_csv("../data/transcript_spell_checked.csv")

In [12]:
df, emotions = prepare_data(df, "text", "labels")

Detected dataset type: goemotions
Starting batch preprocessing...
✓ Text cleaning completed
✓ All NLP processing completed


In [13]:
df_test, emotions_test = prepare_data(df_test, "Translation", "emotion_final")

Detected dataset type: transcript
Starting batch preprocessing...
✓ Text cleaning completed
✓ All NLP processing completed


In [14]:
df_test

,Start Time,End Time,Sentence,Translation,Emotion_fine,Emotion_core,Intensity,sentence_length,Language,predicted_emotion,Sentence_corrected,ekman_emotion,tokenized_text,lemmatized_text
0,00:00:00,00:00:07,Program zawiera tresci nieodpowiednie dla widz...,The program contains content inappropriate for...,warning,fear,mild,14,formal,disgust,Program zawiera treści nieodpowiednie dla widz...,1,"[program, contains, content, inappropriate, un...","[program, contain, content, inappropriate, und..."
1,00:00:07,00:00:09,Ogladasz na wlasna odpowiedzialnosc.,You watch at your own risk.,caution,fear,mild,4,informal,fear,oglądasz na własną odpowiedzialność.,2,"[watch, risk]","[watch, risk]"
2,00:00:10,00:00:13,Jedziemy do Piekowa!,We're going to Pięków!,excitement,happiness,moderate,3,informal,happiness,Jedziemy do piękowa!,3,"[going, pikw]","[go, pikw]"
3,00:00:14,00:00:17,"Krakowiacek jeden, a okolicow siedem.","One Krakowiacek, and seven surroundings.",pride,happiness,mild,5,informal,neutral,"Krakowiacek jeden, a okolic siedem.",3,"[krakowiacek, seven, surroundings]","[krakowiacek, seven, surrounding]"
4,00:00:19,00:00:22,"Tak go dzgnalem w ta tarcze i widzialem, ze si...",So I stabbed him in that shield and I saw that...,irritation,anger,moderate,11,informal,anger,"tak go dźgnąłem w tą tarczę i widziałem, że si...",0,"[stabbed, shield, saw, got, pissed]","[stab, shield, see, get, piss]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
909,1900-01-01 00:22:21.985986,1900-01-01 00:22:25.985986,"Chcialem codziennie z nimi usiasc, codziennie ...","I wanted to sit with them every day, to have w...",longing,sadness,moderate,15,informal,neutral,"chciałem codziennie z nimi usiąść, codziennie ...",4,"[wanted, sit, day, wine, day, wine]","[want, sit, day, wine, day, wine]"
910,1900-01-01 00:22:25.985986,1900-01-01 00:22:27.985986,Cokolwiek w ogole porozmawiac.,Anything at all to talk about.,resignation,sadness,mild,4,informal,disgust,cokolwiek w ogóle porozmawiać.,4,[talk],[talk]
911,1900-01-01 00:22:27.985986,1900-01-01 00:22:31.985986,"Na wszystko, co nam nie pasuje powiedziec sobi...","For everything that we don't like, to say it t...",assertiveness,neutral,mild,18,informal,disgust,"Na wszystko, co nam nie pasuje powiedzieć sobi...",6,"[nt, like, face, eyes, like, trybson, today]","[not, like, face, eye, like, trybson, today]"
912,1900-01-01 00:22:31.985986,1900-01-01 00:22:32.985986,To jest genialne.,This is brilliant.,admiration,happiness,moderate,3,informal,happiness,To jest genialne.,3,[brilliant],[brilliant]


In [15]:
target_samples = 2500

classes_to_augment = df["ekman_emotion"].value_counts()[df["ekman_emotion"].value_counts() < target_samples].index.tolist()
print(f"Classes to augment: {classes_to_augment}")

Classes to augment: [2, 1]


In [16]:
augmented_df = augment_minority_classes(df, target_samples=target_samples, classes_to_augment=classes_to_augment)
df_augmented = pd.concat([df, augmented_df]).reset_index(drop=True)
print(df_augmented["ekman_emotion"].value_counts())
print(f"Original size: {len(df)}, Augmented size: {len(df_augmented)}")

Augmenting emotion 2: 691 → 2500
Augmenting emotion 1: 638 → 2500
Augmented Sample 1: r hard ll trouble
Augmented Sample 2: nah rest pretty cringe
Augmented Sample 3: painful ref terrible god damn gg buckeye close score suggest
ekman_emotion
3    16920
6    12823
0     5579
5     4160
4     2599
1     2438
2     2413
Name: count, dtype: int64
Original size: 43410, Augmented size: 46932


In [17]:
embedding_model = api.load("glove-twitter-100")

In [18]:
X_train, y_train = prepare_rnn_data(df_augmented, "lemmatized_text", embedding_model, "ekman_emotion")
X_test, y_test = prepare_rnn_data(df_test, "lemmatized_text", embedding_model, "ekman_emotion")

Preparing RNN data from 46932 samples...
Embedding dimension: 100
Max sequence length: 50


Converting to embeddings: 100%|██████████| 46932/46932 [00:01<00:00, 37860.89it/s]


✓ Final shape: (46932, 50, 100)
✓ OOV rate: 2.01% (5449/271606)
✓ Data type: float64
Preparing RNN data from 914 samples...
Embedding dimension: 100
Max sequence length: 50


Converting to embeddings: 100%|██████████| 914/914 [00:00<00:00, 38887.36it/s]

✓ Final shape: (914, 50, 100)
✓ OOV rate: 2.09% (52/2492)
✓ Data type: float64


In [19]:
X_train.shape, y_train.shape

((46932, 50, 100), (46932,))

In [20]:
X_test.shape, y_test.shape

((914, 50, 100), (914,))

In [21]:
class EmotionDataset(Dataset):
    """PyTorch Dataset for emotion classification"""

    def __init__(self, sequences, labels):
        self.sequences = torch.FloatTensor(sequences)
        self.labels = torch.LongTensor(labels)

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        return self.sequences[idx], self.labels[idx]

In [22]:
class EmotionRNN(nn.Module):
    """LSTM model for emotion classification"""

    def __init__(self, embedding_dim, hidden_dim, num_classes, num_layers, dropout=0.2):
        super(EmotionRNN, self).__init__()

        self.hidden_dim = hidden_dim
        self.num_layers = num_layers

        # RNN layers
        self.rnn = nn.RNN(
            embedding_dim,
            hidden_dim,
            num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=True
        )

        # Classification head
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 4, hidden_dim),  # *2 for bidirectional
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes)
        )

    def forward(self, x):
        batch_size = x.size(0)

        rnn_out, hidden = self.rnn(x)

        # Attention pooling
        attention_weights = torch.softmax(torch.mean(rnn_out, dim=2), dim=1)
        pooled = torch.sum(rnn_out * attention_weights.unsqueeze(2), dim=1)

        # Last hidden states
        forward_hidden = hidden[-2, :, :]
        backward_hidden = hidden[-1, :, :]
        final_hidden = torch.cat((forward_hidden, backward_hidden), dim=1)

        # Combine both representations
        combined = torch.cat((pooled, final_hidden), dim=1)  # Concatenate both

        output = self.dropout(combined)
        output = self.classifier(output)  # You'll need to adjust input size: hidden_dim*4

        return output

In [23]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.1, random_state=42, stratify=y_train, shuffle=True)

X_train = np.array(X_train)
y_train = np.array(y_train)
X_val = np.array(X_val)
y_val = np.array(y_val)

In [24]:
train_dataset = EmotionDataset(X_train, y_train)
val_dataset = EmotionDataset(X_val, y_val)

In [25]:
train_loader = DataLoader(train_dataset, batch_size=250, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=250, shuffle=False)

In [26]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [27]:
num_classes = len(emotions)
epochs = 250
patience = 10
lr_patience = 10
model_save_path = "../models/EmotionRNN/best_model.pth"
num_classes

7

In [28]:
model = EmotionRNN(
        embedding_dim=100,
        hidden_dim=256,
        num_classes=num_classes,
        num_layers=3,
        dropout=0.1
    ).to(device)

In [29]:
# count balanced class weights
from sklearn.utils.class_weight import compute_class_weight
class_weights = compute_class_weight(
        class_weight='balanced',
        classes=np.unique(y_train),
        y=y_train
    )
class_weights = torch.FloatTensor(class_weights).to(device)
class_weights

tensor([1.2018, 2.7502, 2.7781, 0.3962, 2.5797, 1.6116, 0.5229],
       device='cuda:0')

In [30]:
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(model.parameters(), lr=0.0005, weight_decay=1e-4)

In [31]:
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode='min',
        factor=0.5,      # Reduce LR by this factor
        patience=lr_patience,   # Wait this many epochs before reducing
        min_lr=1e-6,         # Don't reduce below this
    )
early_stopping = EarlyStopping(patience=patience, min_delta=0.001, verbose=True)
checkpoint = ModelCheckpoint(
        filepath=model_save_path,
        monitor='val_loss',
        mode='min',
        verbose=True,
        save_best_only=True
    )

In [32]:
model.train()

EmotionRNN(
  (rnn): RNN(100, 256, num_layers=3, batch_first=True, dropout=0.1, bidirectional=True)
  (dropout): Dropout(p=0.1, inplace=False)
  (classifier): Sequential(
    (0): Linear(in_features=1024, out_features=256, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.1, inplace=False)
    (3): Linear(in_features=256, out_features=7, bias=True)
  )
)

In [33]:
history = {
        'train_loss': [],
        'train_acc': [],
        'val_loss': [],
        'val_acc': [],
        'lr': []
    }

print("Starting training with callbacks...")
print(f"Early stopping patience: {patience}")
print(f"LR reduction patience: {lr_patience}")
print(f"Model will be saved to: {model_save_path}")

for epoch in range(epochs):
        # Training phase
        model.train()
        train_loss = 0
        train_correct = 0
        train_total = 0

        pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs}')
        for sequences, labels in pbar:
            sequences, labels = sequences.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(sequences)
            loss = criterion(outputs, labels)
            loss.backward()

            # Gradient clipping (prevents exploding gradients)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            optimizer.step()

            train_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            train_total += labels.size(0)
            train_correct += (predicted == labels).sum().item()

            # Update progress bar
            current_lr = optimizer.param_groups[0]['lr']
            pbar.set_postfix({
                'Loss': f'{train_loss/(pbar.n+1):.4f}',
                'Acc': f'{100*train_correct/train_total:.2f}%',
                'LR': f'{current_lr:.2e}'
            })

        # Validation phase
        model.eval()
        val_loss = 0
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            for sequences, labels in val_loader:
                sequences, labels = sequences.to(device), labels.to(device)
                outputs = model(sequences)
                loss = criterion(outputs, labels)

                val_loss += loss.item()
                _, predicted = torch.max(outputs, 1)
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()

        # Calculate epoch metrics
        epoch_train_loss = train_loss / len(train_loader)
        epoch_train_acc = 100 * train_correct / train_total
        epoch_val_loss = val_loss / len(val_loader)
        epoch_val_acc = 100 * val_correct / val_total
        current_lr = optimizer.param_groups[0]['lr']

        # Store history
        history['train_loss'].append(epoch_train_loss)
        history['train_acc'].append(epoch_train_acc)
        history['val_loss'].append(epoch_val_loss)
        history['val_acc'].append(epoch_val_acc)
        history['lr'].append(current_lr)

        print(f"Epoch {epoch+1}: Train Loss: {epoch_train_loss:.4f}, Train Acc: {epoch_train_acc:.2f}%, "
              f"Val Loss: {epoch_val_loss:.4f}, Val Acc: {epoch_val_acc:.2f}%, LR: {current_lr:.2e}")

        # APPLY CALLBACKS
        # 1. Learning Rate Scheduler
        scheduler.step(epoch_val_loss)

        # 2. Model Checkpoint (save best model)
        checkpoint(model, epoch_val_loss, epoch, optimizer, {
            'train_acc': epoch_train_acc,
            'val_acc': epoch_val_acc,
            'num_classes': num_classes,
            'embedding_dim': 100
        })

        # 3. Early Stopping
        early_stopping(epoch_val_loss)
        if early_stopping.early_stop:
            print(f"Early stopping triggered at epoch {epoch+1}")
            break

Starting training with callbacks...
Early stopping patience: 10
LR reduction patience: 10
Model will be saved to: ../models/EmotionRNN/best_model.pth


Epoch 1/250: 100%|██████████| 169/169 [00:01<00:00, 99.40it/s, Loss=1.7489, Acc=38.54%, LR=5.00e-04] 


Epoch 1: Train Loss: 1.6351, Train Acc: 38.54%, Val Loss: 1.5172, Val Acc: 43.12%, LR: 5.00e-04
Saving model checkpoint at epoch 1 with val_loss: 1.5172


Epoch 2/250: 100%|██████████| 169/169 [00:01<00:00, 116.68it/s, Loss=1.4613, Acc=44.74%, LR=5.00e-04]


Epoch 2: Train Loss: 1.4613, Train Acc: 44.74%, Val Loss: 1.4382, Val Acc: 47.04%, LR: 5.00e-04
Saving model checkpoint at epoch 2 with val_loss: 1.4382


...

Epoch 20: Train Loss: 0.7870, Train Acc: 60.74%, Val Loss: 1.2540, Val Acc: 54.54%, LR: 5.00e-04
EarlyStopping counter: 8 out of 10


Epoch 21/250: 100%|██████████| 169/169 [00:01<00:00, 98.51it/s, Loss=0.8126, Acc=62.14%, LR=5.00e-04] 


Epoch 21: Train Loss: 0.7597, Train Acc: 62.14%, Val Loss: 1.3412, Val Acc: 55.82%, LR: 5.00e-04
EarlyStopping counter: 9 out of 10


Epoch 22/250: 100%|██████████| 169/169 [00:01<00:00, 115.55it/s, Loss=0.7364, Acc=62.42%, LR=5.00e-04]


Epoch 22: Train Loss: 0.7364, Train Acc: 62.42%, Val Loss: 1.3191, Val Acc: 55.33%, LR: 5.00e-04
EarlyStopping counter: 10 out of 10
Early stopping triggered at epoch 22


In [34]:
# test
test_dataset = EmotionDataset(X_test, y_test)
test_loader = DataLoader(test_dataset, batch_size=250, shuffle=False)

In [35]:
def load_best_model(checkpoint_path, model_class, device, **model_kwargs):
    checkpoint = torch.load(checkpoint_path, map_location=device)

    model = model_class(**model_kwargs).to(device)
    model.load_state_dict(checkpoint['model_state_dict'])

    print(f"Loaded model from epoch {checkpoint['epoch']+1}")
    print(f"Best validation loss: {checkpoint['best_score']:.4f}")

    return model

In [36]:
best_model = load_best_model(
    checkpoint_path="../models/EmotionRNN/best_model.pth",
    model_class=EmotionRNN,  # Your fixed model class
    device=device,
    embedding_dim=100,
    hidden_dim=256,
    num_classes=7,
    num_layers=3,
    dropout=0.1
)

Loaded model from epoch 12
Best validation loss: 1.1790


In [37]:
from sklearn.metrics import accuracy_score


def evaluate_model(model, data_loader, device, emotion_names):
    model.eval()
    all_predictions = []
    all_labels = []

    with torch.no_grad():
        for sequences, labels in tqdm(data_loader, desc="Evaluating"):
            sequences, labels = sequences.to(device), labels.to(device)
            outputs = model(sequences)
            _, predicted = torch.max(outputs, 1)

            all_predictions.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    accuracy = accuracy_score(all_labels, all_predictions)

    print(f"Accuracy: {accuracy:.4f}")
    print("\nClassification Report:")
    print(classification_report(all_labels, all_predictions, target_names=emotion_names, digits=3, zero_division=0))

    return all_predictions, all_labels

In [38]:
emotion_names = ['anger', 'disgust', 'fear', 'joy', 'sadness', 'surprise', 'neutral']

In [39]:
test_preds, test_true = evaluate_model(best_model, test_loader, device, emotion_names)

Evaluating: 100%|██████████| 4/4 [00:00<00:00, 228.42it/s]

Accuracy: 0.4103

Classification Report:
              precision    recall  f1-score   support

       anger      0.667     0.520     0.584       150
     disgust      0.238     0.114     0.154        44
        fear      0.321     0.220     0.261        41
         joy      0.803     0.271     0.405       225
     sadness      0.431     0.191     0.265       115
    surprise      0.152     0.247     0.188        81
     neutral      0.368     0.698     0.482       258

    accuracy                          0.410       914
   macro avg      0.426     0.323     0.334       914
weighted avg      0.504     0.410     0.401       914

